<a href="https://colab.research.google.com/github/AliAI11/DolphinMind/blob/main/notebooks/baseline_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers bitsandbytes accelerate rich psutil \
    sentence-transformers faiss-cpu rouge-score scikit-learn

print("all dependencies installed")

all dependencies installed


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import psutil
import time
from rich.console import Console
from rich.table import Table
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

console = Console()
console.print("imports successful")

imports successful

In [3]:
# loading the model
model_name = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

console.print(f"Loading {model_name} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

console.print("model loaded")

Loading Qwen/Qwen2.5-3B-Instruct in 4-bit...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model loaded

In [4]:
# checking ram usage
def profile():
    ram = psutil.virtual_memory().used / 1e9
    console.print(f"RAM Used: {ram:.2f} GB")

profile()

RAM Used: 5.05 GB

In [5]:
# testing model
def ask(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

response = ask("What is a character in the bleach anime")
console.print(f"Test Answer: {response}")

Test Answer: What is a character in the bleach anime?

In the Bleach anime, characters are numerous and diverse, each with their own unique personalities, backgrounds, 
and powers. Some of the most popular characters include:

1. Ichigo Kurosaki - The protagonist and main character

2. Rukia Kuchiki - A powerful Soul Reaper who becomes Ichigo's ally

3. Grimmjow Jaegerjaquez - A cold-hearted Espada with a sad past

4. Uryū Ishida - A genius Quincy with a

In [6]:
print("\nLoading text from Project Gutenberg...")

import urllib.request

# The Odyssey by Homer
url = "https://www.gutenberg.org/files/1727/1727-0.txt"

try:
    with urllib.request.urlopen(url) as response:
        long_context = response.read().decode('utf-8')

    # Clean up (remove Gutenberg header/footer)
    start_marker = "*** START OF"
    end_marker = "*** END OF"

    if start_marker in long_context:
        long_context = long_context.split(start_marker)[1]
    if end_marker in long_context:
        long_context = long_context.split(end_marker)[0]

    # Make it longer by repeating
    long_context = long_context * 2

    print("Downloaded The Odyssey")
    print(f"Words: {len(long_context.split()):,}")
    print(f"Estimated tokens: {len(tokenizer.encode(long_context)):,}")

except Exception as e:
    print(f"Failed to download: {e}")
    print("Using fallback text")
    # Fallback
    long_context = """Machine learning is a branch of artificial intelligence.""" * 1000

# Test queries about the Odyssey
test_queries = [
    "What is the Telemachy?",
    "Who is Polyphemos?",
    "What happened to Odysseus during his wanderings?"
]

reference_answers = [
    "The Telemachy is the first four books of the Odyssey focusing on Telemachos.",
    "Polyphemos is a Cyclops, son of Poseidon, who was blinded by Odysseus.",
    "Odysseus wandered for years after the Trojan War facing various challenges."
]

print(f"Created {len(test_queries)} test queries")


Loading text from Project Gutenberg...
Downloaded The Odyssey
Words: 259,156


Token indices sequence length is longer than the specified maximum sequence length for this model (351054 > 131072). Running this sequence through the model will result in indexing errors


Estimated tokens: 351,054
Created 3 test queries


In [7]:
print("\n=== BASELINE 1: Truncated Context ===")

def baseline_truncated(context, query, max_tokens=4000):
    """Simply truncate context to fit in window."""
    context_tokens = tokenizer.encode(context)[:max_tokens]
    truncated_context = tokenizer.decode(context_tokens, skip_special_tokens=True)

    prompt = f"Context: {truncated_context}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer1 = baseline_truncated(long_context, test_queries[0])
time1 = time.time() - start

print(f"Answer: {answer1}")
print(f"Time: {time1:.2f}s\n")




=== BASELINE 1: Truncated Context ===
Testing: What is the Telemachy?
Answer: The Telemachy refers to the portion of the "Odyssey" that deals with the story of Telemachus, the son of Odysseus, who sets out on a journey to Pylos to seek news of his father. This section of the epic begins with Book 1, Line 80, and continues until Book 4, and it is not resumed until Odysseus wakes in the middle of Book 13, Line 187. The
Time: 24.18s



In [8]:
print("=== BASELINE 2: Naive Chunking (TF-IDF) ===")

def baseline_naive_chunking(context, query, chunk_size=500, top_k=3):
    """Split into chunks and retrieve with TF-IDF."""
    words = context.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    print(f"Created {len(chunks)} chunks")

    vectorizer = TfidfVectorizer()
    chunk_vectors = vectorizer.fit_transform(chunks)
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    relevant_chunks = [chunks[i] for i in top_indices]
    combined = '\n\n'.join(relevant_chunks)

    prompt = f"Context: {combined}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer2 = baseline_naive_chunking(long_context, test_queries[0])
time2 = time.time() - start

print(f"Answer: {answer2}")
print(f"Time: {time2:.2f}s\n")


=== BASELINE 2: Naive Chunking (TF-IDF) ===
Testing: What is the Telemachy?
Created 519 chunks
Answer: The Telemachy refers to the section of Homer's epic poem "Odyssey" that focuses on the journey and adventures of Telemachus, the son of Odysseus, as he embarks on a quest to seek news of his absent father. This part of the Odyssey explores Telemachus's growth and development as he navigates the challenges of adulthood and the responsibilities that come with being a son of a famous hero. The Telemachy serves as a bridge between
Time: 12.54s



In [16]:
print("=== BASELINE 3: DolphinMind RAG (Overlapping Chunks) ===")
# Load embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embedding_model = embedding_model.cpu()
print("Loaded\n")

def dolphinmind_rag(context, query, chunk_size=500, overlap=100, top_k=5):
    """DolphinMind: Overlapping chunks + FAISS semantic search."""
    words = context.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += (chunk_size - overlap)
        if end >= len(words):
            break

    print(f"Created {len(chunks)} overlapping chunks")

    embeddings = embedding_model.encode(chunks, show_progress_bar=False)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    faiss.normalize_L2(embeddings)
    index.add(embeddings)

    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)

    print(f"Retrieved top {top_k} relevant chunks")

    relevant_chunks = [chunks[idx] for idx in indices[0]]
    combined = '\n\n'.join(relevant_chunks)

    prompt = f"Context: {combined}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer3 = dolphinmind_rag(long_context, test_queries[0])
time3 = time.time() - start

print(f"Answer: {answer3}")
print(f"Time: {time3:.2f}s\n")

=== BASELINE 3: DolphinMind RAG (Overlapping Chunks) ===
Loading embedding model...
Loaded

Testing: What is the Telemachy?
Created 648 overlapping chunks
Retrieved top 5 relevant chunks
Answer: The Telemachy refers to the narrative section of Homer's Odyssey in which Telemachus, Odysseus' son, embarks on a journey to find out the whereabouts of his absent father. This section of the epic poem chronicles Telemachus' adventures and encounters, including his visit to the court of Nestor in Pylos and his subsequent travels to Sparta. It serves as a bridge between the events of the Odyssey and the eventual reunion of Telemachus
Time: 141.60s



In [10]:
print("\n=== EVALUATING ALL BASELINES ===\n")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

results = []

for method_name, method_func in [
    ("Truncated", baseline_truncated),
    ("Naive Chunking", baseline_naive_chunking),
    ("DolphinMind RAG", dolphinmind_rag)
]:
    print(f"Evaluating {method_name}...")

    predictions = []
    times = []

    for i, (query, ref) in enumerate(zip(test_queries, reference_answers)):
        print(f"  Query {i+1}/{len(test_queries)}: {query[:50]}...")
        start = time.time()
        pred = method_func(long_context, query)
        times.append(time.time() - start)
        predictions.append(pred)

    rouge_scores = [
        scorer.score(ref, pred)['rougeL'].fmeasure
        for pred, ref in zip(predictions, reference_answers)
    ]
    avg_rouge = np.mean(rouge_scores)
    avg_time = np.mean(times)

    results.append({
        'method': method_name,
        'rouge_l': avg_rouge,
        'avg_time': avg_time
    })

    print(f"  ROUGE-L: {avg_rouge:.3f}")
    print(f"  Avg Time: {avg_time:.2f}s\n")

print("Evaluation complete")


=== EVALUATING ALL BASELINES ===

Evaluating Truncated...
  Query 1/3: What is the Telemachy?...
  Query 2/3: Who is Polyphemos?...
  Query 3/3: What happened to Odysseus during his wanderings?...
  ROUGE-L: 0.128
  Avg Time: 14.63s

Evaluating Naive Chunking...
  Query 1/3: What is the Telemachy?...
Created 519 chunks
  Query 2/3: Who is Polyphemos?...
Created 519 chunks
  Query 3/3: What happened to Odysseus during his wanderings?...
Created 519 chunks
  ROUGE-L: 0.132
  Avg Time: 8.79s

Evaluating DolphinMind RAG...
  Query 1/3: What is the Telemachy?...
Created 648 overlapping chunks
Retrieved top 5 relevant chunks
  Query 2/3: Who is Polyphemos?...
Created 648 overlapping chunks
Retrieved top 5 relevant chunks
  Query 3/3: What happened to Odysseus during his wanderings?...
Created 648 overlapping chunks
Retrieved top 5 relevant chunks
  ROUGE-L: 0.174
  Avg Time: 58.67s

Evaluation complete


In [11]:
print("\n=== FINAL RESULTS ===\n")

# Simple text table
print(f"{'Method':<25} {'ROUGE-L':>10} {'Avg Time (s)':>15}")
print("-" * 52)

for r in results:
    print(f"{r['method']:<25} {r['rouge_l']:>10.3f} {r['avg_time']:>15.2f}")

print("-" * 52)

# Find best
best = max(results, key=lambda x: x['rouge_l'])
fastest = min(results, key=lambda x: x['avg_time'])

print(f"\nBest ROUGE-L: {best['method']} ({best['rouge_l']:.3f})")
print(f"Fastest: {fastest['method']} ({fastest['avg_time']:.2f}s)")

profile()


=== FINAL RESULTS ===

Method                       ROUGE-L    Avg Time (s)
----------------------------------------------------
Truncated                      0.128           14.63
Naive Chunking                 0.132            8.79
DolphinMind RAG                0.174           58.67
----------------------------------------------------

Best ROUGE-L: DolphinMind RAG (0.174)
Fastest: Naive Chunking (8.79s)


RAM Used: 5.90 GB

In [12]:
import json

results_dict = {
    'experiment': 'baseline_comparison',
    'model': model_name,
    'context_length': len(long_context.split()),
    'num_queries': len(test_queries),
    'results': results
}

with open('baseline_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print("\nResults saved to baseline_results.json")



Results saved to baseline_results.json


In [15]:
print("\n===SUMMARY ===\n")
print(f"tested 3 baseline methods")
print(f"context length: {len(long_context.split()):,} words")
print(f"evaluated on {len(test_queries)} queries")
print(f"memory usage: {psutil.virtual_memory().used / 1e9:.2f} GB")



===SUMMARY ===

tested 3 baseline methods
context length: 259,156 words
evaluated on 3 queries
memory usage: 4.72 GB
